In [21]:
import torch
device = torch.device("cpu")

In [22]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SimpleCNN(nn.Module):
    def __init__(self, width=64):
        super().__init__()

        self.conv1 = nn.Conv2d(3, width, 3, padding=1)
        self.conv2 = nn.Conv2d(width, width, 3, padding=1)
        self.conv3 = nn.Conv2d(width, width, 3, padding=1)

        self.fc1 = nn.Linear(width * 4 * 4, 256)
        self.fc2 = nn.Linear(256, 10)

        self.pool = nn.MaxPool2d(2,2)

    def forward(self,x):

        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))

        x = x.view(x.size(0),-1)

        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return x

In [23]:
from tqdm import tqdm

def train(model, loader, optimizer, device):

    model.train()
    loss_fn = nn.CrossEntropyLoss()

    for x,y in loader:

        x,y = x.to(device), y.to(device)

        optimizer.zero_grad()

        out = model(x)
        loss = loss_fn(out,y)

        loss.backward()
        optimizer.step()

In [24]:
def evaluate(model, loader, device):

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for x,y in loader:

            x,y = x.to(device),y.to(device)

            out = model(x)
            pred = out.argmax(dim=1)

            correct += (pred==y).sum().item()
            total += y.size(0)

    return correct/total

In [25]:
def prune_by_percentile(model, mask_dict, percent):

    for name, param in model.named_parameters():

        if 'weight' in name:

            tensor = param.data.cpu().numpy()
            mask = mask_dict[name]

            alive = tensor[mask==1]

            percentile_value = abs(alive).flatten()
            threshold = torch.quantile(
                torch.tensor(percentile_value), percent
            )

            new_mask = torch.where(
                abs(param.data) < threshold,
                torch.zeros_like(param),
                torch.ones_like(param)
            )

            mask_dict[name] = mask * new_mask

In [26]:
def apply_mask(model, mask_dict):

    for name, param in model.named_parameters():

        if name in mask_dict:
            param.data *= mask_dict[name]

In [27]:
def lottery_ticket_experiment(model, train_loader, test_loader, device,
                              pruning_rounds=10, prune_percent=0.2):

    initial_state = {k:v.clone() for k,v in model.state_dict().items()}

    mask_dict = {}
    for name,param in model.named_parameters():
        mask_dict[name] = torch.ones_like(param)

    results = []

    for round in range(pruning_rounds):

        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

        for epoch in range(10):
            train(model, train_loader, optimizer, device)

        acc = evaluate(model, test_loader, device)
        print("round", round, "acc", acc)

        results.append(acc)

        prune_by_percentile(model, mask_dict, prune_percent)

        # rewind weights
        model.load_state_dict(initial_state)

        apply_mask(model, mask_dict)

    return results, mask_dict

In [28]:
import torchvision
import torchvision.transforms as transforms

def load_cifar10(batch_size=128):

    transform = transforms.Compose([
        transforms.ToTensor()
    ])

    train = torchvision.datasets.CIFAR10(
        root='./data', train=True, download=True, transform=transform
    )

    test = torchvision.datasets.CIFAR10(
        root='./data', train=False, download=True, transform=transform
    )

    train_loader = torch.utils.data.DataLoader(train,batch_size=batch_size,shuffle=True)
    test_loader = torch.utils.data.DataLoader(test,batch_size=batch_size)

    return train_loader, test_loader

In [29]:
def transfer_ticket(mask_dict, width, device):

    model = SimpleCNN(width).to(device)

    apply_mask(model, mask_dict)

    train_loader, test_loader = load_cifar100()

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    for epoch in range(10):
        train(model, train_loader, optimizer, device)

    acc = evaluate(model, test_loader, device)

    print("Transfer accuracy:", acc)

In [30]:
widths = [16,32,64,128,256]

results = {}

for w in widths:

    print("width:",w)

    model = SimpleCNN(width=w).to(device)

    train_loader,test_loader = load_cifar10()

    res,mask = lottery_ticket_experiment(
        model,
        train_loader,
        test_loader,
        device
    )

    results[w] = res

width: 16
round 0 acc 0.6262
round 1 acc 0.6287
round 2 acc 0.6392
round 3 acc 0.6328
round 4 acc 0.6467
round 5 acc 0.6562
round 6 acc 0.6332
round 7 acc 0.6464
round 8 acc 0.6303
round 9 acc 0.6181
width: 32
round 0 acc 0.7009
round 1 acc 0.708
round 2 acc 0.7092
round 3 acc 0.7185
round 4 acc 0.707
round 5 acc 0.7008
round 6 acc 0.6957
round 7 acc 0.6953
round 8 acc 0.6796
round 9 acc 0.681
width: 64
round 0 acc 0.7322
round 1 acc 0.73
round 2 acc 0.7414
round 3 acc 0.7252
round 4 acc 0.722
round 5 acc 0.7244
round 6 acc 0.6933
round 7 acc 0.7113
round 8 acc 0.7008
round 9 acc 0.7001
width: 128
round 0 acc 0.7479
round 1 acc 0.7602


KeyboardInterrupt: 

In [31]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms

###########################################
# GPU setup
###########################################

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True

BATCH = 512
WIDTH = 64

###########################################
# Model
###########################################

class CNN(nn.Module):

    def __init__(self, width=64, classes=10):
        super().__init__()

        self.c1 = nn.Conv2d(3,width,3,padding=1)
        self.c2 = nn.Conv2d(width,width,3,padding=1)
        self.c3 = nn.Conv2d(width,width,3,padding=1)

        self.pool = nn.MaxPool2d(2)

        self.f1 = nn.Linear(width*4*4,256)
        self.f2 = nn.Linear(256,classes)

    def forward(self,x):

        x=self.pool(F.relu(self.c1(x)))
        x=self.pool(F.relu(self.c2(x)))
        x=self.pool(F.relu(self.c3(x)))

        x=torch.flatten(x,1)

        x=F.relu(self.f1(x))
        x=self.f2(x)

        return x


###########################################
# Data
###########################################

transform = transforms.ToTensor()

c10_train = torchvision.datasets.CIFAR10(
    "./data",train=True,download=True,transform=transform)

c10_test = torchvision.datasets.CIFAR10(
    "./data",train=False,download=True,transform=transform)

c100_train = torchvision.datasets.CIFAR100(
    "./data",train=True,download=True,transform=transform)

c100_test = torchvision.datasets.CIFAR100(
    "./data",train=False,download=True,transform=transform)

loader10 = torch.utils.data.DataLoader(
        c10_train,
        batch_size=BATCH,
        shuffle=True,
        num_workers=12,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=4
    )
test10 = torch.utils.data.DataLoader(
        c10_test,
        batch_size=BATCH,
        num_workers=12,
        pin_memory=True,
        persistent_workers=True
    )

loader100 = torch.utils.data.DataLoader(
        c100_train,
        batch_size=BATCH,
        shuffle=True,
        num_workers=12,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=4
    )

test100 = torch.utils.data.DataLoader(
        c100_test,
        batch_size=BATCH,
        num_workers=12,
        pin_memory=True,
        persistent_workers=True
    )

###########################################
# Train + evaluate
###########################################

def train_epoch(model,loader,opt):

    loss_fn=nn.CrossEntropyLoss()
    model.train()

    for x,y in loader:

        x=x.to(device)
        y=y.to(device)

        opt.zero_grad()

        out=model(x)
        loss=loss_fn(out,y)

        loss.backward()
        opt.step()


def evaluate(model,loader):

    model.eval()
    correct=0
    total=0

    with torch.no_grad():

        for x,y in loader:

            x=x.to(device)
            y=y.to(device)

            pred=model(x).argmax(1)

            correct+=(pred==y).sum().item()
            total+=y.size(0)

    return correct/total


###########################################
# Mask utilities
###########################################

def make_mask(model):

    return {n:torch.ones_like(p)
            for n,p in model.named_parameters()}


def apply_mask(model,mask):

    for n,p in model.named_parameters():

        if n in mask:
            p.data.mul_(mask[n])


def global_prune(model,mask,percent):

    weights=[]

    for n,p in model.named_parameters():

        if "weight" in n:

            alive=p.data[mask[n].bool()].abs()
            weights.append(alive)

    thresh=torch.quantile(torch.cat(weights),percent)

    for n,p in model.named_parameters():

        if "weight" in n:

            new_mask=(p.data.abs()>thresh).float()
            mask[n]=mask[n]*new_mask


###########################################
# 1. Find ticket on CIFAR10
###########################################

print("\nFinding CIFAR10 ticket")

model=CNN(WIDTH,10).to(device)

initial={k:v.clone() for k,v in model.state_dict().items()}
mask=make_mask(model)

for r in range(5):

    opt=torch.optim.Adam(model.parameters(),lr=1e-3)

    for e in range(2):
        train_epoch(model,loader10,opt)

    acc=evaluate(model,test10)

    print("round",r,"acc",acc)

    global_prune(model,mask,0.2)

    model.load_state_dict(initial)
    apply_mask(model,mask)


###########################################
# 2. Transfer mask to CIFAR100
###########################################

print("\nTransfer mask to CIFAR100")

model_transfer=CNN(WIDTH,100).to(device)
apply_mask(model_transfer,mask)

opt=torch.optim.Adam(model_transfer.parameters(),lr=1e-3)

for e in range(10):

    train_epoch(model_transfer,loader100,opt)

    acc=evaluate(model_transfer,test100)

    print("transfer epoch",e,"acc",acc)


###########################################
# 3. Dense CIFAR100 baseline
###########################################

print("\nDense CIFAR100 baseline")

dense=CNN(WIDTH,100).to(device)

opt=torch.optim.Adam(dense.parameters(),lr=1e-3)

for e in range(10):

    train_epoch(dense,loader100,opt)

    acc=evaluate(dense,test100)

    print("dense epoch",e,"acc",acc)


###########################################
# 4. Random mask baseline
###########################################

print("\nRandom mask baseline")

rand=CNN(WIDTH,100).to(device)

rand_mask=make_mask(rand)

for n,p in rand_mask.items():

    rand_mask[n]=(torch.rand_like(p)>0.8).float()

apply_mask(rand,rand_mask)

opt=torch.optim.Adam(rand.parameters(),lr=1e-3)

for e in range(10):

    train_epoch(rand,loader100,opt)

    acc=evaluate(rand,test100)

    print("random mask epoch",e,"acc",acc)


Finding CIFAR10 ticket
round 0 acc 0.4936
round 1 acc 0.5021
round 2 acc 0.521
round 3 acc 0.5355
round 4 acc 0.5433

Transfer mask to CIFAR100


RuntimeError: The size of tensor a (100) must match the size of tensor b (10) at non-singleton dimension 0